# Project Infra UI/UX model — Kaggle QLoRA fine-tuning

This notebook fine-tunes `unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit` on a balanced mixture of the project's merged JSONL, a capped WebSight screenshot-to-HTML subset, and a capped Rico UI-element grounding subset. It accepts the three schemas currently present in `final_dataset/merged.jsonl`: generated `prompt`/`response` rows, completed generator rows with `final`, and chat-format rows with `messages`. Failed or empty records are discarded.

The visual path is enabled automatically when records contain images. The external datasets are deliberately capped so the project-specific behavior is not drowned out and the run remains suitable for 2×T4 GPUs.

In [ ]:
# Kaggle: turn Internet on before running this cell.
!pip install -q -U unsloth unsloth_zoo transformers trl peft datasets bitsandbytes
# The notebook does not use audio. Kaggle can ship torchaudio built for a
# different CUDA version than PyTorch; transformers probes it on import.
!pip uninstall -y torchaudio


In [ ]:
from pathlib import Path
import os

# Upload merged.jsonl as a Kaggle Dataset and change this path if needed.
DATA_PATH = Path('/kaggle/input/uiux-model-dataset/merged.jsonl')
if not DATA_PATH.exists():
    candidates = list(Path('/kaggle/input').glob('**/merged.jsonl'))
    if candidates:
        DATA_PATH = candidates[0]
    else:
        # Useful when this notebook is run from the repository itself.
        DATA_PATH = Path('/kaggle/working/merged.jsonl')

MODEL_NAME = 'unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit'
OUTPUT_DIR = '/kaggle/working/uiux-qwen3-vl-lora'
MAX_SEQ_LENGTH = 3072
# Conservative caps prevent full-resolution screenshots from exhausting Kaggle RAM.
WEBSIGHT_LIMIT = 1500
RICO_LIMIT = 500
MAX_IMAGE_SIDE = 768
CUSTOM_REPEAT = 4
MAX_STEPS = 1200
USE_EXTERNAL_DATASETS = True
SEED = 3407

print('Dataset path:', DATA_PATH)
print('Exists:', DATA_PATH.exists())

In [ ]:
import base64, hashlib, json, random
from io import BytesIO
from PIL import Image
from datasets import Dataset

random.seed(SEED)

def as_text(value):
    if value is None:
        return ''
    if isinstance(value, str):
        return value.strip()
    return json.dumps(value, ensure_ascii=False, indent=2)

def decode_image(value):
    if not value:
        return None
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if isinstance(value, dict):
        value = value.get('data') or value.get('bytes') or value.get('image')
    if isinstance(value, (bytes, bytearray)):
        try:
            return Image.open(BytesIO(value)).convert('RGB')
        except Exception:
            return None
    if not isinstance(value, str):
        return None
    try:
        if len(value) < 500 and Path(value).exists():
            return Image.open(value).convert('RGB')
        raw = value.split(',', 1)[1] if value.startswith('data:') else value
        return Image.open(BytesIO(base64.b64decode(raw))).convert('RGB')
    except Exception:
        return None

def normalise_record(row):
    # Already-normalized conversational examples.
    if isinstance(row.get('messages'), list):
        messages = row['messages']
        return {'messages': messages, 'has_image': any(
            isinstance(m.get('content'), list) and any(c.get('type') == 'image' for c in m['content'] if isinstance(c, dict))
            for m in messages if isinstance(m, dict)
        )}

    if row.get('prompt') and row.get('response') is not None:
        user_text = as_text(row['prompt'])
        answer = as_text(row['response'])
    elif row.get('status') == 'done' and row.get('final'):
        app = row.get('app_type', 'digital product')
        style = row.get('style', 'appropriate')
        user_text = f'Design and explain a high-quality UI/UX for a {app} in a {style} style. Include layout, interaction, responsive behavior, accessibility, design tokens, states, and implementation-ready details.'
        answer = row['final']
    else:
        return None
    if len(user_text) < 8 or len(answer) < 20:
        return None
    image = decode_image(row.get('screenshot_b64') or row.get('image'))
    content = [{'type': 'text', 'text': user_text}]
    if image is not None:
        content.insert(0, {'type': 'image', 'image': image})
    return {'messages': [
        {'role': 'user', 'content': content},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]},
    ], 'has_image': image is not None}

rows, rejected = [], 0
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        try:
            item = normalise_record(json.loads(line))
        except Exception:
            item = None
        if item is None:
            rejected += 1
        else:
            rows.append(item)

# Exact duplicate conversations waste capacity and can bias the adapter.
unique, seen = [], set()
for item in rows:
    fingerprint = hashlib.sha256(json.dumps(item['messages'], ensure_ascii=False, default=str).encode()).hexdigest()
    if fingerprint not in seen:
        seen.add(fingerprint); unique.append(item)
deduped = len(rows) - len(unique)
rows = unique
custom_rows = rows
print({'raw_rows': sum(1 for _ in DATA_PATH.open(encoding='utf-8')), 'usable_custom_rows': len(custom_rows), 'rejected': rejected, 'duplicates_removed': deduped})

In [ ]:
# Download capped, task-relevant external data. Kaggle Internet must be enabled.
from datasets import load_dataset

def first_present(row, *keys):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return None

def external_example(image, prompt, answer):
    image = decode_image(image)
    if image is None or len(as_text(answer)) < 20:
        return None
    image = image.convert('RGB')
    image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
    return {'messages': [{'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}
    ]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': as_text(answer)}]}], 'has_image': True}

websight_rows, rico_rows = [], []
if USE_EXTERNAL_DATASETS:
    # WebSight: screenshot -> HTML/CSS. The conservative cap keeps decoded images
    # within Kaggle RAM while still providing a useful code prior.
    web_stream = load_dataset('HuggingFaceM4/WebSight', 'v0.2', split='train', streaming=True)
    for item in web_stream: 
        image = first_present(item, 'image', 'images')
        html = first_present(item, 'text', 'html', 'code')
        ex = external_example(image, 'Recreate this webpage as a responsive, accessible HTML/CSS/SVG implementation. Return only the implementation code.', html)
        if ex is not None:
            websight_rows.append(ex)
        if len(websight_rows) >= WEBSIGHT_LIMIT:
            break

    # Rico: screenshot + view hierarchy -> structured visual grounding. We turn the
    # annotations into an auxiliary answer format instead of mixing raw detection rows
    # into website-code answers.
    rico_stream = load_dataset('Voxel51/rico', split='train', streaming=True)
    for item in rico_stream:
        image = first_present(item, 'image', 'img', 'filepath')
        detections = item.get('detections', {})
        if isinstance(detections, dict):
            detections = detections.get('detections', [])
        if not isinstance(detections, list) or not detections:
            continue
        elements = []
        for det in detections[:80]:
            if not isinstance(det, dict):
                continue
            box = det.get('bounding_box', det.get('bbox'))
            label = det.get('label') or det.get('type') or det.get('content_or_function') or 'ui_element'
            if box is not None:
                elements.append({'label': label, 'bbox': box, 'clickable': det.get('clickable')})
        if not elements:
            continue
        answer = json.dumps({'task': 'UI element grounding', 'elements': elements}, ensure_ascii=False)
        ex = external_example(image, 'Inspect this mobile UI screenshot and return the visible UI elements with their labels, bounding boxes, and clickability. Preserve the supplied JSON schema.', answer)
        if ex is not None:
            rico_rows.append(ex)
        if len(rico_rows) >= RICO_LIMIT:
            break

# Oversample project-specific behavior, then add the capped auxiliary data.
rows = custom_rows * CUSTOM_REPEAT + websight_rows + rico_rows
random.shuffle(rows)
split = max(1, int(len(rows) * 0.10))
eval_rows, train_rows = rows[:split], rows[split:]
train_ds, eval_ds = Dataset.from_list(train_rows), Dataset.from_list(eval_rows)
has_images = any(x['has_image'] for x in rows)
print({'custom': len(custom_rows), 'custom_after_repeat': len(custom_rows) * CUSTOM_REPEAT, 'websight': len(websight_rows), 'rico': len(rico_rows), 'train': len(train_ds), 'validation': len(eval_ds), 'has_images': has_images})

In [ ]:
# Inspect a normalized example before spending GPU time.
example = train_ds[0]
print(example['messages'][0]['content'][0].get('text', '')[:1000])
print('Answer preview:', example['messages'][1]['content'][0]['text'][:1000])
print('Images present:', example['has_image'])

## Load the base model and attach QLoRA

The capped WebSight and Rico subsets provide image supervision. Vision adapters are enabled automatically when image records are present; this lets the model learn visual grounding while the larger language adapter learns diagnosis and code generation.

In [ ]:
import torch
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
    max_seq_length=MAX_SEQ_LENGTH,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=has_images,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
print('Vision adapters enabled:', has_images)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        optim='adamw_8bit',
        logging_steps=5,
        eval_strategy='steps',
        eval_steps=25,
        save_strategy='steps',
        save_steps=25,
        save_total_limit=2,
        bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
        fp16=not (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8),
        report_to='none',
        remove_unused_columns=False,
        seed=SEED,
    ),
)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# Train. For a quick smoke test, temporarily set max_steps=10 in SFTConfig above.
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter to', OUTPUT_DIR)
print(train_result.metrics)

In [ ]:
# Save the exact normalized split used for reproducibility.
train_ds.to_json('/kaggle/working/uiux_train_normalized.jsonl')
eval_ds.to_json('/kaggle/working/uiux_eval_normalized.jsonl')

# Basic inference smoke test (text-only prompt works even when the dataset has images).
from transformers import TextStreamer
FastVisionModel.for_inference(model)
messages = [{'role': 'user', 'content': [{'type': 'text', 'text': 'Review a dashboard UI for hierarchy, accessibility, responsive behavior, and interaction quality. Return concise, actionable recommendations.'}]}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to('cuda')
_ = model.generate(inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, streamer=TextStreamer(tokenizer, skip_prompt=True))

## Dataset and training notes

- The notebook downloads at most 1,500 WebSight examples and 500 Rico examples, resizes them to 768px maximum, then repeats the project's custom rows four times. This keeps project behavior prominent while avoiding Kaggle RAM exhaustion.
- The default run is capped at 1,200 optimizer steps, one epoch, 3,072 tokens, and batch size 1 with gradient accumulation 8. This is a practical starting point for a 2×T4 / under-10-hour budget, but benchmark the first 50 steps on your Kaggle session before committing to the full run.
- Add more completed `rate`, `improve`, and `recreate` records with screenshots from the generator's optional screenshot stage. Those are the most important examples for your deployed product.
- Upload `merged.jsonl` as a Kaggle Dataset. WebSight and Rico are downloaded by the notebook when Kaggle Internet is enabled; you do not need to pre-upload those large source datasets.
- Tavily, URL fetching, screenshot rendering, and HTML execution belong in the application layer at inference time, not inside the fine-tuned weights.